In [1]:
# %pip install pandas numpy
# %pip install scikit-learn
# %pip install RDKit
# %pip install matplotlib networkx
# %pip install tqdm 
# %pip install rdkit
# %pip install xgboost
# %pip install sklearn
# %pip install lightgbm


In [2]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (recall_score, accuracy_score, confusion_matrix)
from sklearn.base import clone
from lightgbm import LGBMClassifier
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from tqdm import tqdm

In [3]:
## load in dataset
tox_data = pd.read_csv('../data/tox21.csv')
tox_data.head(2)

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O


In [4]:
nbt = 2048
threshold = 0.5

# 1. Data Preprocessing  
### Missing Values  
- **Fill with 1**: Prefer false positives (toxic) over false negatives (non-toxic) ❌ Not ideal in practice  

- **Skip missing values?**  
  Filter out missing labels for each tag individually ✅ Ensures model robustness, especially when many labels are missing

In [5]:
# Step 1: Identify label columns (12 toxicity-related labels)
label_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]

# Step 2: Count the number of samples with completely missing or non-missing labels
total_samples = tox_data.shape[0]
all_nan_labels = tox_data[label_cols].isna().all(axis=1).sum()
no_nan_labels = tox_data[label_cols].notna().all(axis=1).sum()
partial_nan_labels = total_samples - all_nan_labels - no_nan_labels

{
    "Total number of samples": total_samples,
    "Samples with all labels missing": int(all_nan_labels),
    "Samples with no missing labels": int(no_nan_labels),
    "Samples with partially missing labels": int(partial_nan_labels)
}

# Count the number of missing values for each label
missing_counts_per_label = tox_data[label_cols].isna().sum().sort_values(ascending=False)

missing_counts_per_label

# ## Option: Fill missing label values with 1 (assume absence of toxicity feature)
# # Fill all missing values in label columns with 1, indicating "non-toxic"
# tox_data[label_cols] = tox_data[label_cols].fillna(1)

# # Check whether the filling was successful (should return 0)
# remaining_missing = tox_data[label_cols].isna().sum().sum()
# int(remaining_missing)


SR-MMP           2092
SR-ARE           2078
NR-Aromatase     2072
NR-ER            1697
NR-PPAR-gamma    1430
SR-HSE           1419
NR-AhR           1322
NR-AR-LBD        1111
SR-p53           1104
NR-ER-LBD         901
SR-ATAD5          781
NR-AR             574
dtype: int64

# 2. Feature Engineering 
- (Molecular Fingerprint Extraction)  
### Extract ECFP4 fingerprints from each SMILES using RDKit:

- Set radius=2 to generate ECFP4 fingerprints  
- Specify a fixed vector length, e.g., nBits=1024  
- Each molecule is converted into a sparse binary vector of 0s and 1s  

### Final feature matrix shape: (num_samples, nbt)

In [6]:
# from rdkit import Chem
# from rdkit.Chem import AllChem
# from rdkit import DataStructs


# # Define a function to convert SMILES to ECFP4 fingerprint vector
# def smiles_to_ecfp4(smiles, radius=2, nBits=nbt):
#     mol = Chem.MolFromSmiles(smiles)  # Use RDKit to convert SMILES into a molecule object (Mol)
#     if mol is None:
#         return np.zeros(nBits, dtype=int)  # Return an all-zero vector if SMILES is invalid

#     fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)  
#     # Generate ECFP4 fingerprint; radius=2 means ECFP4, nBits specifies fingerprint length (e.g., 1024 bits)

#     arr = np.zeros((nBits,), dtype=int)  # Create a one-dimensional array of all zeros
#     DataStructs.ConvertToNumpyArray(fp, arr)  # Copy the fingerprint BitVect to the NumPy array
#     return arr  # Return a NumPy binary vector (0/1) with length nBits



def smiles_to_ecfp4(smiles, radius=2, nBits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nBits, dtype=int)
    
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)
    fp = gen.GetFingerprint(mol)

    arr = np.zeros((nBits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


# Apply the fingerprint extraction with progress bar
tqdm.pandas()
X_fp = np.vstack(tox_data['smiles'].progress_apply(smiles_to_ecfp4))

# Label matrix
y = tox_data[label_cols].values

# Output the shapes of the resulting matrices
print("Feature matrix X_fp.shape =", X_fp.shape)
print("Label matrix y.shape =", y.shape)


100%|██████████| 8006/8006 [00:00<00:00, 11038.80it/s]

Feature matrix X_fp.shape = (8006, 1024)
Label matrix y.shape = (8006, 12)


## 📍 Step 3：Models
### Binary Relevance Strategy 
- The problem is decomposed into 12 independent binary classification tasks


In [7]:
def recall_with_custom_threshold(threshold):
    def scorer(estimator, X, y):
        y_proba = estimator.predict_proba(X)[:, 1]
        y_pred = (y_proba > threshold).astype(int)
        return recall_score(y, y_pred)
    return scorer


def auc_with_proba():
    def scorer(estimator, X, y):
        y_proba = estimator.predict_proba(X)[:, 1]
        return roc_auc_score(y, y_proba)
    return scorer


def custom_auc(y_true, y_proba):
    if len(np.unique(y_true)) == 1:
        return 0.5  # fallback for constant labels
    return roc_auc_score(y_true, y_proba)

auc_scorer = make_scorer(custom_auc, needs_proba=True)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_scorer.py:548: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


### 1.  XGBoost (Gradient Boosting)
- The problem is decomposed into 12 independent binary classification tasks  
- An individual XGBoost model is trained for each label

### 1.threshold = 0.5
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.7624</td>
      <td>0.2376</td>
      <td>0.4904</td>
      <td>0.7624</td>
      <td>0.5969</td>
      <td>0.9222</td>
      <td>0.9262</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.7045</td>
      <td>0.2955</td>
      <td>0.4973</td>
      <td>0.7045</td>
      <td>0.5831</td>
      <td>0.8876</td>
      <td>0.8832</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.7812</td>
      <td>0.2188</td>
      <td>0.5208</td>
      <td>0.7812</td>
      <td>0.6250</td>
      <td>0.9782</td>
      <td>0.8491</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.5926</td>
      <td>0.4074</td>
      <td>0.3019</td>
      <td>0.5926</td>
      <td>0.4000</td>
      <td>0.9668</td>
      <td>0.8335</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.7561</td>
      <td>0.2439</td>
      <td>0.4366</td>
      <td>0.7561</td>
      <td>0.5536</td>
      <td>0.9648</td>
      <td>0.8171</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.6471</td>
      <td>0.3529</td>
      <td>0.2558</td>
      <td>0.6471</td>
      <td>0.3667</td>
      <td>0.9450</td>
      <td>0.8089</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.5000</td>
      <td>0.5000</td>
      <td>0.1316</td>
      <td>0.5000</td>
      <td>0.2083</td>
      <td>0.9423</td>
      <td>0.8081</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.5772</td>
      <td>0.4228</td>
      <td>0.3698</td>
      <td>0.5772</td>
      <td>0.4508</td>
      <td>0.8541</td>
      <td>0.7877</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.5814</td>
      <td>0.4186</td>
      <td>0.4032</td>
      <td>0.5814</td>
      <td>0.4762</td>
      <td>0.9630</td>
      <td>0.7683</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.5385</td>
      <td>0.4615</td>
      <td>0.1842</td>
      <td>0.5385</td>
      <td>0.2745</td>
      <td>0.9719</td>
      <td>0.7328</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.5185</td>
      <td>0.4815</td>
      <td>0.2295</td>
      <td>0.5185</td>
      <td>0.3182</td>
      <td>0.9495</td>
      <td>0.7286</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.5469</td>
      <td>0.4531</td>
      <td>0.2201</td>
      <td>0.5469</td>
      <td>0.3139</td>
      <td>0.8788</td>
      <td>0.6720</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.6255</td>
      <td>0.3745</td>
      <td>0.3368</td>
      <td>0.6255</td>
      <td>0.4306</td>
      <td>0.9354</td>
      <td>0.8013</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.5870</td>
      <td>0.4130</td>
      <td>0.3358</td>
      <td>0.5870</td>
      <td>0.4254</td>
      <td>0.9472</td>
      <td>0.8085</td>
    </tr>
  </tbody>
</table>
</div>

### b.threshold
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.6528</td>
      <td>0.3472</td>
      <td>0.5987</td>
      <td>0.6528</td>
      <td>0.6246</td>
      <td>0.9155</td>
      <td>0.9210</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.6685</td>
      <td>0.3315</td>
      <td>0.6578</td>
      <td>0.6685</td>
      <td>0.6631</td>
      <td>0.8943</td>
      <td>0.8952</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.7407</td>
      <td>0.2593</td>
      <td>0.3774</td>
      <td>0.7407</td>
      <td>0.5000</td>
      <td>0.9723</td>
      <td>0.8446</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.7353</td>
      <td>0.2647</td>
      <td>0.5208</td>
      <td>0.7353</td>
      <td>0.6098</td>
      <td>0.9768</td>
      <td>0.8031</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.3333</td>
      <td>0.6667</td>
      <td>0.0921</td>
      <td>0.3333</td>
      <td>0.1443</td>
      <td>0.9370</td>
      <td>0.7821</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.4785</td>
      <td>0.5215</td>
      <td>0.4635</td>
      <td>0.4785</td>
      <td>0.4709</td>
      <td>0.8314</td>
      <td>0.7760</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.5306</td>
      <td>0.4694</td>
      <td>0.3023</td>
      <td>0.5306</td>
      <td>0.3852</td>
      <td>0.9399</td>
      <td>0.7757</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.6957</td>
      <td>0.3043</td>
      <td>0.4507</td>
      <td>0.6957</td>
      <td>0.5470</td>
      <td>0.9627</td>
      <td>0.7753</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.5000</td>
      <td>0.5000</td>
      <td>0.2295</td>
      <td>0.5000</td>
      <td>0.3146</td>
      <td>0.9486</td>
      <td>0.7716</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.6579</td>
      <td>0.3421</td>
      <td>0.4032</td>
      <td>0.6579</td>
      <td>0.5000</td>
      <td>0.9664</td>
      <td>0.7417</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.6667</td>
      <td>0.3333</td>
      <td>0.2632</td>
      <td>0.6667</td>
      <td>0.3774</td>
      <td>0.9749</td>
      <td>0.7182</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.4653</td>
      <td>0.5347</td>
      <td>0.2956</td>
      <td>0.4653</td>
      <td>0.3615</td>
      <td>0.8685</td>
      <td>0.6743</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.5938</td>
      <td>0.4062</td>
      <td>0.3879</td>
      <td>0.5938</td>
      <td>0.4582</td>
      <td>0.9324</td>
      <td>0.7899</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.6554</td>
      <td>0.3446</td>
      <td>0.3903</td>
      <td>0.6554</td>
      <td>0.4854</td>
      <td>0.9442</td>
      <td>0.7758</td>
    </tr>
  </tbody>
</table>
</div>



### a.Grid Search  
- Applied to the label found to be the least accurate: `"SR-ATAD5"`

In [8]:
# Set target label
target_label = "SR-ATAD5"

# Filter samples with non-missing values for the target label
valid_idx = ~tox_data[target_label].isna()
X_valid = X_fp[valid_idx]
y_valid = tox_data[target_label].values[valid_idx]

print(f"🚀 Starting hyperparameter tuning for {target_label}, valid samples: {len(y_valid)}")

# Split into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(
    X_valid, y_valid, test_size=0.2, random_state=5104, stratify=y_valid
)

🚀 Starting hyperparameter tuning for SR-ATAD5, valid samples: 7225


In [9]:
param_grid = {
    'max_depth': [5, 10 , 15],
    'n_estimators': [500 , 700 , 900],
    'learning_rate': [0.05, 0.1 ,0.12]
}



# GridSearchCV 
grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss'),
    param_grid,
    # scoring= recall_with_custom_threshold(threshold),
    scoring= auc_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)

# GridSearch
grid.fit(X_train, y_train)

# best parameters
print("\n🎯 best parameter:")
print(grid.best_params_)

y_pred = grid.best_estimator_.predict(X_test)

# print the auc on the testset

print(f"\n📈 Recall on testset: {recall_score(y_test, y_pred):.4f}")

Fitting 3 folds for each of 27 candidates, totalling 81 fits

🎯 best parameter:
{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 500}

📈 Recall on testset: 0.2075


### b.Fit and Predict
- Fit the model using the best parameters and make predictions.

In [10]:
X = X_fp
y = tox_data[label_cols].values
label_names = label_cols

In [11]:
from sklearn.metrics import (
    confusion_matrix, recall_score, accuracy_score,
    precision_score, roc_auc_score, f1_score
)

# Initialize containers
models = {}
metrics = {}

# Collect predictions and truths for overall evaluation
all_y_true = []
all_y_pred = []
all_y_proba = []

for label in label_names:
    valid_idx = ~tox_data[label].isna()
    X_valid = X_fp[valid_idx]
    y_valid = tox_data[label].values[valid_idx]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid,
        test_size=0.2,
        random_state=5104,
        stratify=y_valid
    )

    # model = clone(grid.best_estimator_)
    # model.set_params(verbosity=0)  # Silence XGBoost output
    # model.fit(X_train, y_train)
    
    # best parameter got from grid search
    best_parameter = {
    'n_estimators': 500,
    'max_depth': 5,
    'learning_rate': 0.05,
    'random_state': 5104,
    'verbosity': 0}

    model = XGBClassifier(**best_parameter)
    model.fit(X_train, y_train)
    
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    positive_rate = np.mean(y_test)
    tp_rate = tp / (tp + fp) if (tp + fp) > 0 else 0
    fp_rate = fp / (tp + fp) if (tp + fp) > 0 else 0
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    models[label] = model
    metrics[label] = {
        "Positive Rate": round(positive_rate, 4),
        "TP Rate": round(tp_rate, 4),
        "FP Rate": round(fp_rate, 4),
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "F1 Score": round(f1, 4),
        "Accuracy": round(accuracy, 4),
        "AUC": round(auc, 4)
    }

    # Collect for overall metrics
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_proba.extend(y_proba)

# Convert metrics to DataFrame and sort
eval_df = pd.DataFrame(metrics).T.sort_values(by="AUC", ascending=False)

# Calculate mean and median of all metrics across labels
overall_mean = eval_df.astype(float).mean().round(4).to_dict()
overall_median = eval_df.astype(float).median().round(4).to_dict()

# Add two summary rows: mean and median across all labels
eval_df.loc["Overall (mean)"] = overall_mean
eval_df.loc["Overall (median)"] = overall_median

# Display the final evaluation table
display(eval_df)



,Positive Rate,TP Rate,FP Rate,Recall,Precision,F1 Score,Accuracy,AUC
NR-AhR,0.1174,0.7660,0.2340,0.4586,0.7660,0.5737,0.9200,0.9173
SR-MMP,0.1581,0.7611,0.2389,0.4599,0.7611,0.5733,0.8918,0.8918
NR-AR-LBD,0.0348,0.8148,0.1852,0.4583,0.8148,0.5867,0.9775,0.8334
SR-ATAD5,0.0367,0.7333,0.2667,0.2075,0.7333,0.3235,0.9682,0.8321
NR-ER-LBD,0.0500,0.8214,0.1786,0.3239,0.8214,0.4646,0.9627,0.7941
SR-p53,0.0623,0.9333,0.0667,0.1628,0.9333,0.2772,0.9471,0.7880
SR-ARE,0.1619,0.6316,0.3684,0.2500,0.6316,0.3582,0.8550,0.7804
NR-Aromatase,0.0514,0.7500,0.2500,0.1967,0.7500,0.3117,0.9553,0.7765
SR-HSE,0.0577,0.6667,0.3333,0.0526,0.6667,0.0976,0.9439,0.7735
NR-AR,0.0417,0.8333,0.1667,0.4032,0.8333,0.5435,0.9718,0.7674


### 2. LightGBM


### a.threshold = 0.5
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.7308</td>
      <td>0.2692</td>
      <td>0.3631</td>
      <td>0.7308</td>
      <td>0.4851</td>
      <td>0.9095</td>
      <td>0.8821</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.7353</td>
      <td>0.2647</td>
      <td>0.4011</td>
      <td>0.7353</td>
      <td>0.5190</td>
      <td>0.8825</td>
      <td>0.8406</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.8000</td>
      <td>0.2000</td>
      <td>0.1395</td>
      <td>0.8000</td>
      <td>0.2376</td>
      <td>0.9442</td>
      <td>0.8375</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.7500</td>
      <td>0.2500</td>
      <td>0.1698</td>
      <td>0.7500</td>
      <td>0.2769</td>
      <td>0.9675</td>
      <td>0.8313</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.8065</td>
      <td>0.1935</td>
      <td>0.5208</td>
      <td>0.8065</td>
      <td>0.6329</td>
      <td>0.9790</td>
      <td>0.8197</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.7619</td>
      <td>0.2381</td>
      <td>0.2623</td>
      <td>0.7619</td>
      <td>0.3902</td>
      <td>0.9579</td>
      <td>0.7969</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.7667</td>
      <td>0.2333</td>
      <td>0.2396</td>
      <td>0.7667</td>
      <td>0.3651</td>
      <td>0.8651</td>
      <td>0.7885</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.7586</td>
      <td>0.2414</td>
      <td>0.3099</td>
      <td>0.7586</td>
      <td>0.4400</td>
      <td>0.9606</td>
      <td>0.7827</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.7500</td>
      <td>0.2500</td>
      <td>0.1579</td>
      <td>0.7500</td>
      <td>0.2609</td>
      <td>0.9742</td>
      <td>0.7793</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.7083</td>
      <td>0.2917</td>
      <td>0.2237</td>
      <td>0.7083</td>
      <td>0.3400</td>
      <td>0.9499</td>
      <td>0.7376</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.8462</td>
      <td>0.1538</td>
      <td>0.3548</td>
      <td>0.8462</td>
      <td>0.5000</td>
      <td>0.9704</td>
      <td>0.7118</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.7083</td>
      <td>0.2917</td>
      <td>0.2138</td>
      <td>0.7083</td>
      <td>0.3285</td>
      <td>0.8899</td>
      <td>0.7006</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.7602</td>
      <td>0.2398</td>
      <td>0.2797</td>
      <td>0.7602</td>
      <td>0.3980</td>
      <td>0.9376</td>
      <td>0.7924</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.7543</td>
      <td>0.2457</td>
      <td>0.2510</td>
      <td>0.7543</td>
      <td>0.3776</td>
      <td>0.9539</td>
      <td>0.7927</td>
    </tr>
  </tbody>
</table>
</div>


### b.threshold = 0.3
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Positive Rate</th>
      <th>TP Rate</th>
      <th>FP Rate</th>
      <th>Recall</th>
      <th>Precision</th>
      <th>F1 Score</th>
      <th>Accuracy</th>
      <th>AUC</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>NR-AhR</th>
      <td>0.1174</td>
      <td>0.5915</td>
      <td>0.4085</td>
      <td>0.5350</td>
      <td>0.5915</td>
      <td>0.5619</td>
      <td>0.9020</td>
      <td>0.8811</td>
    </tr>
    <tr>
      <th>SR-MMP</th>
      <td>0.1581</td>
      <td>0.5778</td>
      <td>0.4222</td>
      <td>0.5561</td>
      <td>0.5778</td>
      <td>0.5668</td>
      <td>0.8656</td>
      <td>0.8540</td>
    </tr>
    <tr>
      <th>NR-AR-LBD</th>
      <td>0.0348</td>
      <td>0.7073</td>
      <td>0.2927</td>
      <td>0.6042</td>
      <td>0.7073</td>
      <td>0.6517</td>
      <td>0.9775</td>
      <td>0.8325</td>
    </tr>
    <tr>
      <th>SR-p53</th>
      <td>0.0623</td>
      <td>0.6154</td>
      <td>0.3846</td>
      <td>0.2791</td>
      <td>0.6154</td>
      <td>0.3840</td>
      <td>0.9442</td>
      <td>0.8325</td>
    </tr>
    <tr>
      <th>SR-ATAD5</th>
      <td>0.0367</td>
      <td>0.7222</td>
      <td>0.2778</td>
      <td>0.2453</td>
      <td>0.7222</td>
      <td>0.3662</td>
      <td>0.9689</td>
      <td>0.8095</td>
    </tr>
    <tr>
      <th>SR-ARE</th>
      <td>0.1619</td>
      <td>0.5500</td>
      <td>0.4500</td>
      <td>0.4583</td>
      <td>0.5500</td>
      <td>0.5000</td>
      <td>0.8516</td>
      <td>0.7907</td>
    </tr>
    <tr>
      <th>NR-Aromatase</th>
      <td>0.0514</td>
      <td>0.7308</td>
      <td>0.2692</td>
      <td>0.3115</td>
      <td>0.7308</td>
      <td>0.4368</td>
      <td>0.9587</td>
      <td>0.7881</td>
    </tr>
    <tr>
      <th>NR-ER-LBD</th>
      <td>0.0500</td>
      <td>0.6190</td>
      <td>0.3810</td>
      <td>0.3662</td>
      <td>0.6190</td>
      <td>0.4602</td>
      <td>0.9571</td>
      <td>0.7690</td>
    </tr>
    <tr>
      <th>NR-PPAR-gamma</th>
      <td>0.0289</td>
      <td>0.5833</td>
      <td>0.4167</td>
      <td>0.1842</td>
      <td>0.5833</td>
      <td>0.2800</td>
      <td>0.9726</td>
      <td>0.7441</td>
    </tr>
    <tr>
      <th>SR-HSE</th>
      <td>0.0577</td>
      <td>0.7000</td>
      <td>0.3000</td>
      <td>0.2763</td>
      <td>0.7000</td>
      <td>0.3962</td>
      <td>0.9514</td>
      <td>0.7316</td>
    </tr>
    <tr>
      <th>NR-ER</th>
      <td>0.1260</td>
      <td>0.4370</td>
      <td>0.5630</td>
      <td>0.3270</td>
      <td>0.4370</td>
      <td>0.3741</td>
      <td>0.8621</td>
      <td>0.6902</td>
    </tr>
    <tr>
      <th>NR-AR</th>
      <td>0.0417</td>
      <td>0.7742</td>
      <td>0.2258</td>
      <td>0.3871</td>
      <td>0.7742</td>
      <td>0.5161</td>
      <td>0.9697</td>
      <td>0.6892</td>
    </tr>
    <tr>
      <th>Overall (mean)</th>
      <td>0.0772</td>
      <td>0.6340</td>
      <td>0.3660</td>
      <td>0.3775</td>
      <td>0.6340</td>
      <td>0.4578</td>
      <td>0.9318</td>
      <td>0.7844</td>
    </tr>
    <tr>
      <th>Overall (median)</th>
      <td>0.0546</td>
      <td>0.6172</td>
      <td>0.3828</td>
      <td>0.3466</td>
      <td>0.6172</td>
      <td>0.4485</td>
      <td>0.9542</td>
      <td>0.7894</td>
    </tr>
  </tbody>
</table>
</div>

### a.Grid Search  
- Applied to the label found to be the least accurate: `"SR-ATAD5"`

In [13]:
# Set target label
target_label = "SR-ATAD5"

# Filter samples with non-missing values for the target label
valid_idx = ~tox_data[target_label].isna()
X_valid = X_fp[valid_idx]
y_valid = tox_data[target_label].values[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_valid, y_valid, test_size=0.2, random_state=5104, stratify=y_valid
)

In [14]:
# # Grid search with LightGBM
# grid = GridSearchCV(
#     LGBMClassifier(eval_metric='logloss', verbosity=-1),  # ✅ Fully suppress logs and warnings
#     param_grid={
#         'n_estimators': [300, 500, 700],
#         'max_depth': [8, 10, 13, 15],
#         'learning_rate': [0.05, 0.1]
#     },
#     # scoring=recall_with_custom_threshold(threshold),
#     scoring=auc_with_proba(),  # ✅ Make sure this function is silent
#     cv=3,
#     verbose=0,  # ✅ Suppress GridSearchCV logs too
#     n_jobs=-1
# )

# # Fit the grid search
# grid.fit(X_train, y_train)

# # Print best parameters
# print("🎯 Best hyperparameters:", grid.best_params_)


### b.Fit and Predict
- Fit the model using the best parameters and make predictions.

In [15]:

models = {}
metrics = {}

# Store all predictions and ground truths across labels (for overall evaluation)
all_y_true = []
all_y_pred = []
all_y_proba = []

for label in label_names:
    # Select samples with non-missing values for the current label
    valid_idx = ~tox_data[label].isna()
    X_valid = X_fp[valid_idx]
    y_valid = tox_data[label].values[valid_idx]

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid, test_size=0.2, random_state=42, stratify=y_valid
    )

    # # model = LGBMClassifier(**best_params)
    # model = clone(grid.best_estimator_)
    # model.set_params(verbosity=-1)  # Optional: suppress LightGBM warnings
    # model.fit(X_train, y_train)
    
    # best_params got from grid search
    best_params = {
    'n_estimators': 300,
    'max_depth': 8,
    'learning_rate': 0.05,
     'random_state': 5104,
    'verbosity': -1 # Optional: suppress LightGBM warnings
    }

    model = LGBMClassifier(**best_params)
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    positive_rate = np.mean(y_test)
    tp_rate = tp / (tp + fp) if (tp + fp) > 0 else 0
    fp_rate = fp / (tp + fp) if (tp + fp) > 0 else 0
    
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    # Store the trained model and evaluation metrics
    models[label] = model
    metrics[label] = {
        "Positive Rate": round(positive_rate, 4),
        "TP Rate": round(tp_rate, 4),
        "FP Rate": round(fp_rate, 4),
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "F1 Score": round(f1, 4),
        "Accuracy": round(accuracy, 4),
        "AUC": round(auc, 4)
    }

    # Collect data for overall evaluation
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_proba.extend(y_proba)



# Convert metrics to DataFrame and sort
eval_df = pd.DataFrame(metrics).T.sort_values(by="AUC", ascending=False)

# Calculate mean and median of all metrics across labels
overall_mean = eval_df.astype(float).mean().round(4).to_dict()
overall_median = eval_df.astype(float).median().round(4).to_dict()

# Add two summary rows: mean and median across all labels
eval_df.loc["Overall (mean)"] = overall_mean
eval_df.loc["Overall (median)"] = overall_median

# Display the final evaluation table
display(eval_df)


,Positive Rate,TP Rate,FP Rate,Recall,Precision,F1 Score,Accuracy,AUC
NR-AhR,0.1174,0.7292,0.2708,0.4459,0.7292,0.5534,0.9155,0.8811
SR-MMP,0.1581,0.6721,0.3279,0.4385,0.6721,0.5307,0.8774,0.8540
NR-AR-LBD,0.0348,0.7222,0.2778,0.5417,0.7222,0.6190,0.9768,0.8325
SR-p53,0.0623,0.8125,0.1875,0.1512,0.8125,0.2549,0.9450,0.8325
SR-ATAD5,0.0367,0.8182,0.1818,0.1698,0.8182,0.2812,0.9682,0.8095
SR-ARE,0.1619,0.7031,0.2969,0.2344,0.7031,0.3516,0.8600,0.7907
NR-Aromatase,0.0514,0.8095,0.1905,0.2787,0.8095,0.4146,0.9596,0.7881
NR-ER-LBD,0.0500,0.6562,0.3438,0.2958,0.6562,0.4078,0.9571,0.7690
NR-PPAR-gamma,0.0289,0.6667,0.3333,0.1579,0.6667,0.2553,0.9734,0.7441
SR-HSE,0.0577,0.7500,0.2500,0.1974,0.7500,0.3125,0.9499,0.7316
